# 32_Ablation_Preprocess: ablation tiền xử lý ảnh trên step2 s1 (mục 5, GĐ3 đến GĐ5)
**Mục đích:** với mỗi cấu hình (bật/tắt affine, Gray-World, khử nhiễu Lee, CLAHE), trích lại landmark từ VIDEO GỐC của 48 video test cũ, nội suy lên 120 fps như lúc làm dữ liệu, rồi chấm step2 s1 (không train lại) bằng đúng độ đo của GĐ1/GĐ2.

**Cấu hình:** 1_goc (không tiền xử lý, trích lại để kiểm tra đường chạy), 2_affine, 3_affine_gw, 4_affine_lee, 5_affine_clahe, 6_all4. Affine là nền vì 3 khối sau chạy trên ảnh cắt 256x256 do affine tạo.

**Đường chạy mỗi khung** (như mô tả trong báo cáo): có landmark khung trước -> affine cắt quanh tay (cạnh cắt 5,5 lần độ dài cổ tay-khớp 9) -> các khối đang bật -> MediaPipe trên ảnh cắt -> đổi landmark về tọa độ ảnh gốc. Khung đầu video hoặc ngay sau khi mất tay: MediaPipe chạy trên ảnh gốc. Nếu MediaPipe không thấy tay trên ảnh cắt thì khung đó tính là mất tay (NaN).
Nội suy 120 fps giữ đúng như 01 và 17: Dataset_hand chèn khung giữa (cell 6 của 01), Dataset_bonus nội suy theo thời gian thật (cell 4 của 17), dataset_labeled giữ nguyên 120 fps.

**Cách chạy:** `C:\envs\train-gpu\Scripts\activate.bat`, kernel train-gpu, chạy lần lượt cell 1 đến 6. Cần `demo_v2/game_logic.py` copy vào cùng thư mục với notebook. Danh sách cấu hình chạy đặt ở `RUN` (cell 6). Landmark đã trích lưu ở `Data_321/ablation_out/<cấu hình>/`, chạy lại thì dùng cache (`FORCE = True` để trích lại).

In [34]:
#cell 1: cấu hình, nạp 48 video test cũ (manifest_all.csv) và tìm video gốc tương ứng
import time, json
from pathlib import Path
from collections import defaultdict
import cv2, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import mediapipe as mp
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision
from game_logic import majority_latest, CLASSES          # game_logic.py để cùng thư mục với notebook

assert torch.cuda.is_available(), "Chưa bật CUDA"
dev = "cuda"
BASE = Path.home() / "Downloads" / "Data_321"
CK = BASE / "checkpoints_task9"
OUT = BASE / "ablation_out"; OUT.mkdir(exist_ok=True)

m = pd.read_csv(BASE / "manifest_all.csv")
T = m[m.tap == "test"].reset_index(drop=True)
assert len(T) == 48, len(T)
T["stem"] = T.path.str.replace("\\", "/", regex=False).str.split("/").str[-1].str.replace(".npz", "", regex=False)

exts = {".mp4", ".mov", ".avi", ".mkv"}
found = defaultdict(list)
for src in T.nguon.unique():
    for p in (BASE / src).rglob("*"):
        if p.suffix.lower() in exts: found[(src, p.stem)].append(p)
miss = [(s, st) for s, st in zip(T.nguon, T.stem) if len(found[(s, st)]) != 1]
assert not miss, f"Không tìm thấy đúng 1 video cho: {miss[:5]} (tổng {len(miss)})"
T["video"] = [found[(s, st)][0] for s, st in zip(T.nguon, T.stem)]
print("Test cũ:", len(T), "video | video gốc tìm thấy:", T.video.notna().sum())
print(T.groupby("nguon").size().to_string())

Test cũ: 48 video | video gốc tìm thấy: 48
nguon
Dataset_bonus      15
Dataset_hand       12
dataset_labeled    21


In [35]:
#cell 2: 4 kỹ thuật xử lý ảnh + affine (sao chép nguyên từ 23_Preprocess cell 2; cạnh cắt đặt 5,5 ở FACTOR, truyền vào affine_matrix)
SIDE, FACTOR = 256, 5.5

def gray_world(img):
    x = img.astype(np.float32)
    mk = x.reshape(-1, 3).mean(0)
    return np.clip(x * (mk.mean() / mk), 0, 255).astype(np.uint8)

def lee_denoise(img, win=5):
    x = img.astype(np.float32)
    m = cv2.blur(x, (win, win))
    v = np.maximum(cv2.blur(x * x, (win, win)) - m * m, 0)
    vn = np.median(v[::4, ::4].reshape(-1, 3), axis=0)
    k = v / (v + vn + 1e-6)
    return np.clip(m + k * (x - m), 0, 255).astype(np.uint8)

def clahe_lab(img, clip=2.0, tile=8):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    lab[:, :, 0] = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile)).apply(lab[:, :, 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

def affine_matrix(lm, W, H, S=256, factor=3.5):
    p0 = lm[0, :2] * (W, H); p9 = lm[9, :2] * (W, H); v = p9 - p0
    d = np.hypot(*v); c = (p0 + p9) / 2
    ang = 90 - np.degrees(np.arctan2(-v[1], v[0]))
    M = cv2.getRotationMatrix2D((float(c[0]), float(c[1])), float(ang), S / (factor * d))
    M[0, 2] += S / 2 - c[0]; M[1, 2] += S / 2 - c[1]
    return M

def affine_warp(img, M, S=256):
    return cv2.warpAffine(img, M, (S, S), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)

def affine_back(lm_crop, M, W, H, S=256):
    Mi = cv2.invertAffineTransform(M); scale = np.hypot(M[0, 0], M[0, 1])
    pts = np.c_[lm_crop[:, 0] * S, lm_crop[:, 1] * S, np.ones(len(lm_crop))] @ Mi.T
    return np.c_[pts[:, 0] / W, pts[:, 1] / H, lm_crop[:, 2] * S / (scale * W)]
print("Cạnh cắt FACTOR =", FACTOR, "| ảnh cắt", SIDE)

Cạnh cắt FACTOR = 5.5 | ảnh cắt 256


In [36]:
#cell 3: đặc trưng 546 và TCN (sao chép nguyên từ 19_Task9_Train cell 2), nạp step2 s1 từ checkpoint, không train
PAIR_I, PAIR_J = torch.triu_indices(21, 21, offset=1, device=dev)

def pos_vel(pos, hand_mask):
    pos = pos.reshape(pos.shape[0], pos.shape[1], 63)
    vel = torch.zeros_like(pos); vel[:, 1:] = (pos[:, 1:] - pos[:, :-1]) * (hand_mask[:, 1:] & hand_mask[:, :-1])[:, :, None]
    return torch.cat([pos, vel], dim=-1)

def features_from_pos(pos, hand_mask):
    pos = torch.where(hand_mask[:, :, None, None], pos, torch.zeros_like(pos))
    pair_valid = (hand_mask[:, 1:] & hand_mask[:, :-1])[:, :, None]
    dist = torch.linalg.vector_norm(pos[:, :, PAIR_I, :] - pos[:, :, PAIR_J, :], dim=-1) * hand_mask[:, :, None]
    dist_vel = torch.zeros_like(dist); dist_vel[:, 1:] = (dist[:, 1:] - dist[:, :-1]) * pair_valid
    return torch.cat([pos_vel(pos, hand_mask), dist, dist_vel], dim=-1)

class Block(nn.Module):
    def __init__(self, c, d):
        super().__init__(); self.pad = 2 * d; self.conv = nn.Conv1d(c, c, 3, dilation=d); self.drop = nn.Dropout(0.1)
    def forward(self, x):
        return x + self.drop(F.relu(self.conv(F.pad(x, (self.pad, 0)))))

class TCN(nn.Module):
    def __init__(self, in_features=126, c=64):
        super().__init__(); self.inp = nn.Linear(in_features, c)
        self.blocks = nn.Sequential(*[Block(c, d) for d in (1, 2, 4, 8, 16)]); self.out = nn.Linear(c, 3)
    def forward(self, x):
        return self.out(self.blocks(self.inp(x).transpose(1, 2)).transpose(1, 2))

p = CK / "step2" / "seed1" / "TCN.pt"; assert p.exists(), p
ck = torch.load(p, map_location=dev, weights_only=False)
net = TCN(546).to(dev); net.load_state_dict(ck["w"]); net.eval()
print("step2 s1 | epoch", ck["epoch"], "| val_loss", round(ck["val_loss"], 4), "| cfg", ck.get("cfg"))

step2 s1 | epoch 27 | val_loss 0.3224 | cfg {'aug': '13_step2', 'n_in': 546, 'variant': 'step2', 'seed': 1, 'epochs': 120, 'patience': 20, 'n_train': 214, 'n_val': 43}


In [37]:
#cell 4: nội suy 120 fps (sao chép nguyên up2 từ 01 cell 6 và up120 từ 17 cell 4) và hàm trích landmark theo cấu hình (2 bộ nhận tay: ảnh gốc / ảnh cắt)
def up2(lm, hand):                       # Dataset_hand: 60 -> 120 fps, chèn khung giữa
    a = torch.tensor(lm, device=dev); h = torch.tensor(hand, device=dev)
    new = torch.empty((2 * len(a) - 1, 21, 3), device=dev)
    new[0::2], new[1::2] = a, (a[:-1] + a[1:]) / 2
    hm = torch.where(h[:-1] == h[1:], h[:-1], torch.full_like(h[:-1], -1))
    nh = torch.empty(2 * len(h) - 1, dtype=h.dtype, device=dev)
    nh[0::2], nh[1::2] = h, hm
    return new.cpu().numpy(), nh.cpu().numpy()

def up120(lm, hand, fps):                # Dataset_bonus: nội suy theo thời gian thật
    a = torch.tensor(lm, device=dev); h = torch.tensor(hand, device=dev); T = len(a)
    t = torch.arange(int((T - 1) / fps * 120 + 1e-6) + 1, device=dev) * (fps / 120)
    i0 = t.floor().long().clamp(max=T - 1); i1 = (i0 + 1).clamp(max=T - 1)
    w = (t - i0).view(-1, 1, 1)
    new = torch.where(w < 1e-6, a[i0], a[i0] + w * (a[i1] - a[i0]))
    nh = torch.where((w.view(-1) < 1e-6) | (h[i0] == h[i1]), h[i0], torch.full_like(h[i0], -1))
    return new.cpu().numpy(), nh.cpu().numpy()

def to120(lm, fps, nguon):               # dataset_labeled quay sẵn 120 fps nên giữ nguyên (như 01)
    h = np.zeros(len(lm), np.int8)       # nhãn tay trái/phải không dùng ở đây
    if nguon == "Dataset_hand": return up2(lm, h)[0]
    if nguon == "Dataset_bonus": return up120(lm, h, fps)[0]
    return lm

OPTS = vision.HandLandmarkerOptions(
    base_options=mp_tasks.BaseOptions(model_asset_path=str(BASE / "hand_landmarker.task")),
    running_mode=vision.RunningMode.VIDEO, num_hands=1)

def extract_cfg(path, steps):
    # trả về landmark (T,21,3) ở tọa độ ảnh gốc (NaN nếu mất tay), fps, và số đo [ms chuẩn bị ảnh, ms MediaPipe, ms đổi tọa độ, số khung, số khung cắt, số khung ảnh gốc, số khung cắt mà mất tay]
    # Hai bộ nhận tay cùng chế độ video: det_o chạy khung ảnh gốc, det_c chạy khung ảnh cắt. Dùng chung 1 bộ thì khung cắt sau khung ảnh gốc
    # gần như luôn mất tay (cell 7, 8: 63% khung cắt mất tay; tách 2 bộ thì 0% trên 8 video thử).
    det_o = vision.HandLandmarker.create_from_options(OPTS)         # tạo mới mỗi video, như 01 và 17
    det_c = vision.HandLandmarker.create_from_options(OPTS) if "affine" in steps else None
    cap = cv2.VideoCapture(str(path)); fps = cap.get(cv2.CAP_PROP_FPS)
    lm, i, prev = [], 0, None
    tp = td = tb = 0.0; n_crop = n_orig = n_fail = 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        H, W = frame.shape[:2]
        crop = (det_c is not None) and prev is not None
        t0 = time.perf_counter()
        if crop:
            M = affine_matrix(prev, W, H, S=SIDE, factor=FACTOR)
            img = affine_warp(frame, M, SIDE)
            if "gw" in steps: img = gray_world(img)
            if "lee" in steps: img = lee_denoise(img)
            if "clahe" in steps: img = clahe_lab(img)
        else:
            img = frame
        t1 = time.perf_counter()
        r = (det_c if crop else det_o).detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(img, cv2.COLOR_BGR2RGB)), int(i * 1000 / fps))
        t2 = time.perf_counter()
        if r.hand_landmarks:
            L = np.array([[q.x, q.y, q.z] for q in r.hand_landmarks[0]], np.float32)
            if crop: L = affine_back(L, M, W, H, SIDE).astype(np.float32)
            prev = L; lm.append(L)
        else:
            lm.append(np.full((21, 3), np.nan, np.float32)); prev = None; n_fail += int(crop)
        t3 = time.perf_counter()
        tp += t1 - t0; td += t2 - t1; tb += t3 - t2
        n_crop += int(crop); n_orig += int(not crop); i += 1
    cap.release(); det_o.close()
    if det_c is not None: det_c.close()
    return np.array(lm, np.float32), fps, np.array([tp * 1000, td * 1000, tb * 1000, i, n_crop, n_orig, n_fail])
print("Đã định nghĩa: up2, up120, to120, extract_cfg (2 bộ nhận tay)")

Đã định nghĩa: up2, up120, to120, extract_cfg (2 bộ nhận tay)


In [38]:
#cell 5: chấm step2 s1 trên landmark 120 fps; độ đo giống 31a và cell 12 của 19 (cử chỉ cuối = đa số trong 0,2 s cuối = 24 khung)
THRS = [0.5, 0.6, 0.7, 0.8, 0.9]
K_END = max(1, round(0.2 * 120))

def lm_to_probs(lm120):                  # cùng cách dựng đặc trưng như 19 cell 1 và 2: trừ cổ tay, chia ‖p9 − p0‖
    with np.errstate(all="ignore"):
        q = lm120 - lm120[:, :1]
        q = q / np.linalg.norm(q[:, 9], axis=1)[:, None, None]
    pos = q.reshape(len(q), 63)
    has = ~np.isnan(pos[:, 0])
    pos = np.nan_to_num(pos).astype(np.float32)
    X = torch.tensor(pos, device=dev).reshape(1, len(pos), 21, 3)
    Mk = torch.tensor(has, device=dev)[None]
    with torch.no_grad(): P = net(features_from_pos(X, Mk)).softmax(-1)[0]
    return P.cpu().numpy(), has

def video_record(P, has, y):
    conf, raw = P.max(-1), P.argmax(-1)
    rec = {"khung": len(has)}
    for tag, thr in (("n", None), ("t50", 0.5), ("t70", 0.7)):      # n = không ngưỡng
        pred = np.where(has & (conf >= (thr or 0)), raw, -1); ok = pred >= 0
        fin = majority_latest([int(v) for v in pred[-K_END:] if v >= 0])
        rec[f"cuoi_{tag}"] = -1 if fin is None else int(fin)
        rec[f"co_{tag}"] = int(ok.sum()); rec[f"dung_{tag}"] = int((ok & (pred == y)).sum())
    for t in THRS: rec[f"v{int(round(t * 100))}"] = bool((has & (raw == y) & (conf >= t)).any())
    return rec

def f1_macro(y, p):
    f = []
    for c in range(3):
        tp = np.sum((y == c) & (p == c)); fp = np.sum((y != c) & (p == c)); fn = np.sum((y == c) & (p != c))
        f.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return float(np.mean(f))

def conf_mat(y, p):
    M = np.zeros((3, 4), int)
    for a, b in zip(y, p): M[int(a), int(b) if b >= 0 else 3] += 1
    return pd.DataFrame(M, index=[f"thật {c}" for c in CLASSES], columns=CLASSES + ["không xác định"])

def summ(D, tag):
    y, p = D.nhan.values, D[f"cuoi_{tag}"].values
    return dict(so_video=len(D),
                acc_moi_khung=round(100 * D[f"dung_{tag}"].sum() / max(D[f"co_{tag}"].sum(), 1), 1),
                ty_le_khung_khong_xd=round(100 * (D.khung.sum() - D[f"co_{tag}"].sum()) / D.khung.sum(), 1),
                acc_cu_chi_cuoi_video=round(100 * np.mean(y == p), 1), f1_macro_cuoi=round(100 * f1_macro(y, p), 1),
                video_khong_xd_cuoi=int((p < 0).sum()),
                video_co_it_nhat_1_khung_dung=round(100 * (D[f"dung_{tag}"] > 0).mean(), 1),
                TB_5_nguong=round(float(np.mean([100 * D[f"v{int(round(t * 100))}"].mean() for t in THRS])), 1))
print("Đã định nghĩa: lm_to_probs, video_record, f1_macro, conf_mat, summ")

Đã định nghĩa: lm_to_probs, video_record, f1_macro, conf_mat, summ


In [39]:
#cell 6: chạy các cấu hình trong RUN. GĐ3: chỉ 1_goc (trích lại, kiểm tra đường chạy) và 2_affine; GĐ4, GĐ5 chỉ đổi danh sách RUN
CONFIGS = {"1_goc": set(), "2_affine": {"affine"}, "3_affine_gw": {"affine", "gw"},
           "4_affine_lee": {"affine", "lee"}, "5_affine_clahe": {"affine", "clahe"}, "6_all4": {"affine", "gw", "lee", "clahe"}}
RUN = ["6_all4"]
FORCE = False

def process_config(name):
    steps = CONFIGS[name]; d_out = OUT / name; d_out.mkdir(exist_ok=True)
    recs, sts, diffs = [], [], []
    for i, r in T.iterrows():
        f = d_out / (r.stem + ".npz")
        if f.exists() and not FORCE:
            z = np.load(f); lm, fps, st = z["lm"], float(z["fps"]), z["st"]
        else:
            lm, fps, st = extract_cfg(r.video, steps); np.savez_compressed(f, lm=lm, fps=fps, st=st)
        lm120 = to120(lm, fps, r.nguon)
        P, has = lm_to_probs(lm120)
        rec = video_record(P, has, int(r.nhan)); rec.update(file=r.stem, nguon=r.nguon, nhan=int(r.nhan))
        recs.append(rec); sts.append(st)
        if name == "1_goc":                                  # đối chiếu với landmark đã lưu ở landmarks_120 (đường chạy phải trùng 01 và 17)
            a = np.load(BASE / r.path)["lm"]
            same = a.shape == lm120.shape
            nan_diff = int((np.isnan(a[:, 0, 0]) != np.isnan(lm120[:, 0, 0])).sum()) if same else -1
            mx = float(np.nanmax(np.abs(a - lm120))) if same and not np.isnan(a).all() else np.nan
            diffs.append((r.stem, same, nan_diff, mx))
        if (i + 1) % 12 == 0: print(f"  {name}: {i + 1}/48 video")
    R = pd.DataFrame(recs); R.to_csv(d_out / "per_video.csv", index=False)
    st = np.sum(sts, axis=0); nf = st[3]
    info = dict(ms_chuan_bi_anh=round(st[0] / nf, 3), ms_mediapipe=round(st[1] / nf, 3), ms_doi_toa_do=round(st[2] / nf, 3),
                ms_tong_moi_khung=round(st[:3].sum() / nf, 3), khung=int(nf), khung_cat=int(st[4]), khung_anh_goc=int(st[5]),
                khung_cat_mat_tay=int(st[6]))
    S = {"tat_ca": summ(R, "n"), **{s: summ(g, "n") for s, g in R.groupby("nguon")}}
    json.dump({"info": info, "n": S, "t50": summ(R, "t50"), "t70": summ(R, "t70")}, open(d_out / "summary.json", "w"), ensure_ascii=False, indent=1)
    print(f"\n===== {name} | không ngưỡng =====")
    print(pd.DataFrame({"tất cả": S["tat_ca"], **{k: v for k, v in S.items() if k != "tat_ca"}}).T.to_string())
    print(conf_mat(R.nhan.values, R.cuoi_n.values).to_string())
    print("ngưỡng 0,5 (tất cả):", summ(R, "t50")); print("ngưỡng 0,7 (tất cả):", summ(R, "t70"))
    print("thời gian và số khung:", info)
    if diffs:
        D = pd.DataFrame(diffs, columns=["video", "cung_kich_thuoc", "khung_lech_NaN", "lech_toi_da"])
        bad = D[(~D.cung_kich_thuoc) | (D.khung_lech_NaN != 0) | (D.lech_toi_da > 1e-4)]
        print(f"ĐỐI CHIẾU với landmarks_120 đã lưu: {len(D) - len(bad)}/48 video trùng (lệch tối đa {D.lech_toi_da.max():.2e}); video lệch:")
        print(bad.to_string() if len(bad) else "không có")

for name in RUN: process_config(name)

  6_all4: 12/48 video
  6_all4: 24/48 video
  6_all4: 36/48 video
  6_all4: 48/48 video

===== 6_all4 | không ngưỡng =====
                 so_video  acc_moi_khung  ty_le_khung_khong_xd  acc_cu_chi_cuoi_video  f1_macro_cuoi  video_khong_xd_cuoi  video_co_it_nhat_1_khung_dung  TB_5_nguong
tất cả               48.0           84.2                   3.3                   93.8           94.5                  1.0                           97.9         95.8
Dataset_bonus        15.0           67.2                   8.4                   86.7           85.6                  0.0                           93.3         88.0
Dataset_hand         12.0           78.6                   2.4                   91.7           95.2                  1.0                          100.0         98.3
dataset_labeled      21.0           99.3                   0.0                  100.0          100.0                  0.0                          100.0        100.0
               paper  rock  scissors  không xác

In [40]:
#cell 7: chẩn đoán mất tay của 2_affine, đọc từ cache ablation_out (không trích lại, không ghi file)
def runs(b):                                   # độ dài các đoạn liên tiếp có tay
    out, n = [], 0
    for v in b:
        if v: n += 1
        elif n: out.append(n); n = 0
    if n: out.append(n)
    return out

rows, all_runs = [], defaultdict(list)
for _, r in T.iterrows():
    h2 = ~np.isnan(np.load(OUT / "2_affine" / (r.stem + ".npz"))["lm"][:, 0, 0])
    h1 = ~np.isnan(np.load(OUT / "1_goc" / (r.stem + ".npz"))["lm"][:, 0, 0])
    crop = np.r_[False, h2[:-1]]               # khung i chạy trên ảnh cắt khi khung i-1 có tay
    lost = crop & ~h2                          # khung cắt mà MediaPipe không thấy tay
    sau_goc = crop & np.r_[False, ~crop[:-1]]  # khung cắt đứng ngay sau 1 khung chạy trên ảnh gốc
    rr = runs(h2); all_runs[r.nguon] += rr
    rows.append(dict(file=r.stem, nguon=r.nguon, khung=len(h2), cat=int(crop.sum()), mat_cat=int(lost.sum()),
                     cat_sau_goc=int(sau_goc.sum()), mat_sau_goc=int((lost & sau_goc).sum()),
                     mat_goc_1=int((~h1).sum()), do_dai_doan_tb=round(float(np.mean(rr)), 1) if rr else 0.0))
D = pd.DataFrame(rows)

S = json.load(open(OUT / "2_affine" / "summary.json"))["info"]
print("Đối chiếu summary.json | khung cắt:", D.cat.sum(), "/", S["khung_cat"], "| mất tay:", D.mat_cat.sum(), "/", S["khung_cat_mat_tay"])

G = D.groupby("nguon")[["khung", "cat", "mat_cat", "cat_sau_goc", "mat_sau_goc", "mat_goc_1"]].sum()
G.loc["tất cả"] = G.sum()
G["%mat_tren_khung_cat"] = (100 * G.mat_cat / G.cat.clip(lower=1)).round(1)
G["%mat_khi_cat_sau_goc"] = (100 * G.mat_sau_goc / G.cat_sau_goc.clip(lower=1)).round(1)
G["%mat_khi_cat_sau_cat"] = (100 * (G.mat_cat - G.mat_sau_goc) / (G.cat - G.cat_sau_goc).clip(lower=1)).round(1)
G["%NaN_cua_1_goc"] = (100 * G.mat_goc_1 / G.khung).round(1)
allr = sum(all_runs.values(), [])
G["do_dai_doan_co_tay_trung_vi"] = [float(np.median(all_runs[k])) for k in G.index[:-1]] + [float(np.median(allr))]
print(G.to_string())

D["%mat_cat"] = (100 * D.mat_cat / D.cat.clip(lower=1)).round(1)
print("\nVideo có >50% khung cắt mất tay:", int((D["%mat_cat"] > 50).sum()), "/ 48")
print(D.sort_values("%mat_cat", ascending=False).head(10)[["file", "nguon", "khung", "cat", "mat_cat", "%mat_cat", "mat_goc_1", "do_dai_doan_tb"]].to_string(index=False))

Đối chiếu summary.json | khung cắt: 4087 / 4087 | mất tay: 5 / 5
                 khung   cat  mat_cat  cat_sau_goc  mat_sau_goc  mat_goc_1  %mat_tren_khung_cat  %mat_khi_cat_sau_goc  %mat_khi_cat_sau_cat  %NaN_cua_1_goc  do_dai_doan_co_tay_trung_vi
nguon                                                                                                                                                                                   
Dataset_bonus      940   918        4           19            1          3                  0.4                   5.3                   0.3             0.3                         62.0
Dataset_hand       763   734        1           12            0         20                  0.1                   0.0                   0.1             2.6                         63.0
dataset_labeled   2456  2435        0           21            0          0                  0.0                   0.0                   0.0             0.0                        105.0
tất cả    

In [41]:
#cell 9: in kết quả cell 8 (sửa lỗi s.cat; dùng lại res và sel đã có trong kernel)
assert "res" in globals() and len(res["hien_tai"]) == len(sel), "Mất res: chạy lại cell 8"
rows = []
for k, v in res.items():
    R = pd.DataFrame(v); s = R[["cat", "mat", "cat_sg", "mat_sg", "cat_sc", "mat_sc"]].sum()
    rows.append(dict(cach_chay=k, khung_cat=int(s["cat"]), khung_mat=int(s["mat"]),
                     pct_mat=round(100 * s["mat"] / max(s["cat"], 1), 1),
                     pct_mat_sau_goc=round(100 * s["mat_sg"] / max(s["cat_sg"], 1), 1),
                     pct_mat_sau_cat=round(100 * s["mat_sc"] / max(s["cat_sc"], 1), 1),
                     lech_xy_TB_so_voi_1_goc=round(float(np.nanmean(R["lech"])), 4)))
print("TỔNG 8 video (chỉ affine):"); print(pd.DataFrame(rows).to_string(index=False))

P = pd.DataFrame({k: [round(100 * x["mat"] / max(x["cat"], 1), 1) for x in v] for k, v in res.items()})
P.insert(0, "file", sel.file.values); P.insert(1, "nguon", sel.nguon.values)
print("\n% khung cắt mất tay theo từng video:"); print(P.to_string(index=False))

TỔNG 8 video (chỉ affine):
    cach_chay  khung_cat  khung_mat  pct_mat  pct_mat_sau_goc  pct_mat_sau_cat  lech_xy_TB_so_voi_1_goc
     hien_tai        386        222     57.5             99.1              0.0                   0.0035
a_hai_bo_nhan        606          0      0.0              0.0              0.0                   0.0015
    b_doc_lap        606          0      0.0              0.0              0.0                   0.0038

% khung cắt mất tay theo từng video:
                 file           nguon  hien_tai  a_hai_bo_nhan  b_doc_lap
VID_20261002075216637    Dataset_hand     100.0            0.0        0.0
VID_20261002080632373    Dataset_hand     100.0            0.0        0.0
      IMG_0510(1)_001 dataset_labeled     100.0            0.0        0.0
      IMG_0510(1)_002 dataset_labeled     100.0            0.0        0.0
VID_20261024173309542   Dataset_bonus     100.0            0.0        0.0
VID_20261025173908825   Dataset_bonus     100.0            0.0        0.0
V

In [42]:
#cell 11: video nào đổi cử chỉ cuối so với 1_goc (đọc per_video.csv của 6 cấu hình, không trích lại)
NAMES = list(CONFIGS)
PV = {n: pd.read_csv(OUT / n / "per_video.csv").set_index("file") for n in NAMES}
base = PV["1_goc"]
assert base.index.is_unique and all(len(v) == 48 and v.index.equals(base.index) for v in PV.values())
def cls(v): return "?" if v < 0 else CLASSES[int(v)]

for tag, ten in (("n", "không ngưỡng"), ("t50", "ngưỡng 0,5"), ("t70", "ngưỡng 0,7")):
    col = f"cuoi_{tag}"
    diff = [f for f in base.index if any(PV[n].loc[f, col] != base.loc[f, col] for n in NAMES)]
    rows = []
    for f in diff:
        row = {"video": f, "nguon": base.loc[f, "nguon"], "that": cls(base.loc[f, "nhan"])}
        for n in NAMES: row[n] = cls(PV[n].loc[f, col])
        rows.append(row)
    print(f"\n=== {ten}: video có cử chỉ cuối khác 1_goc ở ít nhất 1 cấu hình: {len(diff)}/48 ===")
    if rows: print(pd.DataFrame(rows).to_string(index=False))
    print("số video cuối sai:", {n: int((PV[n][col] != PV[n]["nhan"]).sum()) for n in NAMES})


=== không ngưỡng: video có cử chỉ cuối khác 1_goc ở ít nhất 1 cấu hình: 2/48 ===
                video         nguon that 1_goc 2_affine 3_affine_gw 4_affine_lee 5_affine_clahe 6_all4
VID_20261025112806483 Dataset_bonus rock  rock     rock        rock         rock       scissors   rock
VID_20261025174424874 Dataset_bonus rock  rock     rock        rock         rock           rock  paper
số video cuối sai: {'1_goc': 2, '2_affine': 2, '3_affine_gw': 2, '4_affine_lee': 2, '5_affine_clahe': 3, '6_all4': 3}

=== ngưỡng 0,5: video có cử chỉ cuối khác 1_goc ở ít nhất 1 cấu hình: 0/48 ===
số video cuối sai: {'1_goc': 2, '2_affine': 2, '3_affine_gw': 2, '4_affine_lee': 2, '5_affine_clahe': 2, '6_all4': 2}

=== ngưỡng 0,7: video có cử chỉ cuối khác 1_goc ở ít nhất 1 cấu hình: 1/48 ===
                video         nguon that 1_goc 2_affine 3_affine_gw 4_affine_lee 5_affine_clahe 6_all4
VID_20261025174424874 Dataset_bonus rock  rock     rock       paper         rock           rock  paper
số vide

In [43]:
#cell 10: đo thời gian 6 cấu hình trong cùng một lượt (xen kẽ theo từng video), 2 lượt: lượt 1 thứ tự 1->6, lượt 2 đảo 6->1; không dùng cache
ORDER = list(CONFIGS)
extract_cfg(T.video[0], CONFIGS["2_affine"])                      # khởi động, bỏ kết quả
acc = {(p, n): np.zeros(7) for p in (1, 2) for n in ORDER}
for p, order in ((1, ORDER), (2, ORDER[::-1])):
    for i, r in T.iterrows():
        for n in order:
            _, _, st = extract_cfg(r.video, CONFIGS[n]); acc[(p, n)] += st
        if (i + 1) % 12 == 0: print(f"  lượt {p}: {i + 1}/48 video")

rows = []
for n in ORDER:
    s1, s2 = acc[(1, n)], acc[(2, n)]
    f = lambda s, k: s[k] / s[3]
    rows.append(dict(cau_hinh=n,
                     chuan_bi=(f(s1, 0) + f(s2, 0)) / 2, mediapipe=(f(s1, 1) + f(s2, 1)) / 2, doi_toa_do=(f(s1, 2) + f(s2, 2)) / 2,
                     tong_L1=s1[:3].sum() / s1[3], tong_L2=s2[:3].sum() / s2[3],
                     khung_cat=int(s1[4]), khung_mat_tay=int(s1[6])))
Tm = pd.DataFrame(rows)
Tm["tong_TB"] = (Tm.tong_L1 + Tm.tong_L2) / 2
Tm["chenh_giua_2_luot"] = (Tm.tong_L1 - Tm.tong_L2).abs()
Tm = Tm.round(3)
Tm.to_csv(OUT / "timing_interleaved.csv", index=False)
print("Thời gian trung bình mỗi khung (ms), chưa gồm đọc video:"); print(Tm.to_string(index=False))

  lượt 1: 12/48 video
  lượt 1: 24/48 video
  lượt 1: 36/48 video
  lượt 1: 48/48 video
  lượt 2: 12/48 video
  lượt 2: 24/48 video
  lượt 2: 36/48 video
  lượt 2: 48/48 video
Thời gian trung bình mỗi khung (ms), chưa gồm đọc video:
      cau_hinh  chuan_bi  mediapipe  doi_toa_do  tong_L1  tong_L2  khung_cat  khung_mat_tay  tong_TB  chenh_giua_2_luot
         1_goc     0.320     17.587       0.035   14.312   21.571          0              0   17.942              7.259
      2_affine     0.513     13.245       0.181   11.097   16.782       4087              5   13.939              5.685
   3_affine_gw     2.032     13.374       0.178   12.681   18.487       4058             34   15.584              5.806
  4_affine_lee     3.189     13.089       0.178   13.536   19.375       4087              5   16.455              5.840
5_affine_clahe     1.628     14.543       0.185   12.473   20.239       4071             20   16.356              7.767
        6_all4     5.658     14.541       0.183